# Machine Learning — Lab 4
## Linear Regression and Residual Analysis

**Main Course Learning Outcomes — CLO3, CLO4, CLO5**

- **CLO3:** Construct and analyze machine learning models by applying algorithmic principles and internal computations.
- **CLO4:** Apply supervised learning techniques to solve practical problems and interpret their outcomes.
- **CLO5:** Evaluate machine learning models using appropriate metrics and justify decisions based on performance trade-offs and real-world context.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`

> **Assessment principle:** Correct code is not enough. You must connect the model equation, coefficients, predictions, residuals, and evaluation metrics to the real problem.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Problem formulation & baseline | 10 min | Define the regression task and establish a mean baseline |
| 2. Simple linear regression | 25 min | Fit rent from apartment area and interpret slope/intercept |
| 3. Multiple linear regression | 30 min | Add more numerical features and compare performance |
| 4. Manual prediction & residuals | 20 min | Compute predictions and residuals by hand |
| 5. Residual analysis | 20 min | Diagnose error patterns, outliers, and model limitations |
| 6. Challenge, debugging & viva | 15 min | Modify the model and defend conclusions |
| **Total** | **120 min** | |

### Main idea

Linear regression is a complete supervised-learning model:

$$
X
\rightarrow
\hat{y}
\rightarrow
e=y-\hat{y}
\rightarrow
\text{evaluation}.
$$

## Learning Objectives

By the end of this lab, you should be able to:

1. formulate a continuous-target prediction problem as regression;
2. fit and interpret a simple linear regression model;
3. fit a multiple linear regression model;
4. compute a prediction manually from learned coefficients;
5. calculate and interpret residuals;
6. compute MAE, MSE, RMSE, and $R^2$;
7. compare a learned model with a mean baseline;
8. use residual plots to identify systematic model error;
9. explain why low training error does not guarantee generalization;
10. identify one limitation of linear regression from evidence in the data.

# Part I — Apartment Rent Prediction

We will use a synthetic but realistic apartment dataset.

Each row represents one apartment listing.

The target is:

```text
monthly_rent
```

The available input features are:

- `area_m2`
- `bedrooms`
- `building_age`
- `distance_km`
- `furnished`

The data-generating process contains mostly linear effects plus some noise and a mild nonlinear component. That makes the dataset useful for learning both the strengths and limitations of linear regression.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print("Machine Learning Lab 4 environment ready.")

In [ ]:
rng = np.random.default_rng(3452)
n = 360

area_m2 = np.clip(rng.normal(105, 32, n), 40, 220)
bedrooms = np.clip(np.rint(area_m2 / 38 + rng.normal(0, 0.6, n)), 1, 6).astype(int)
building_age = np.clip(rng.gamma(shape=2.0, scale=8.0, size=n), 0, 45)
distance_km = np.clip(rng.gamma(shape=2.3, scale=3.2, size=n), 0.5, 25)
furnished = rng.integers(0, 2, size=n)

# Mostly linear signal, with a mild nonlinear age penalty and random noise.
monthly_rent = (
    650
    + 10.5 * area_m2
    + 260 * bedrooms
    - 11 * building_age
    - 42 * distance_km
    + 320 * furnished
    - 0.28 * (building_age ** 2)
    + rng.normal(0, 220, n)
)

# A few realistic high-end listings create large but valid residuals.
luxury_idx = rng.choice(np.arange(n), size=8, replace=False)
monthly_rent[luxury_idx] += rng.normal(900, 180, size=len(luxury_idx))

df_master = pd.DataFrame({
    "area_m2": np.round(area_m2, 1),
    "bedrooms": bedrooms,
    "building_age": np.round(building_age, 1),
    "distance_km": np.round(distance_km, 1),
    "furnished": furnished,
    "monthly_rent": np.round(monthly_rent, 0),
})

print("Master dataset shape:", df_master.shape)
display(df_master.head())

## Task 1.1 — Personalized Dataset

Enter the **last four digits** of your student ID.

Your ID determines a reproducible sample of 280 apartments.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 4000 + (STUDENT_ID_LAST4 % 6000)

df = df_master.sample(
    n=280,
    random_state=SEED,
    replace=False
).reset_index(drop=True)

print("Your regression seed:", SEED)
print("Working dataset shape:", df.shape)

## Task 1.2 — Formulate the ML Problem

Complete:

- **Learning paradigm:**  
- **Task type:**  
- **One observation represents:**  
- **Target $y$:**  
- **Candidate features $X$:**  
- **Why this is not a classification task:**  
- **What successful generalization means:**  

Then answer:

> Would `monthly_rent` ever be allowed as an input feature for a model that predicts `monthly_rent`? Why not?

# Part II — Train / Validation / Test Split

We will use:

$$
60\% \text{ training},\qquad
20\% \text{ validation},\qquad
20\% \text{ test}.
$$

Unlike classification, ordinary regression does not usually use target stratification directly.

In [ ]:
feature_cols = [
    "area_m2",
    "bedrooms",
    "building_age",
    "distance_km",
    "furnished",
]

X = df[feature_cols].copy()
y = df["monthly_rent"].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=SEED,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

## Task 2.1 — Predict Split Sizes

Before checking the output, calculate the approximate expected counts for 280 rows.

| Split | Expected percentage | Approximate expected count |
|---|---:|---:|
| Training | 60% |  |
| Validation | 20% |  |
| Test | 20% |  |

Then explain why the final test set should not be used to choose which features to include.

# Part III — Mean Baseline

A simple regression baseline predicts the **training-target mean** for every example:

$$
\hat{y}_{baseline}=\bar{y}_{train}.
$$

This gives a reference for deciding whether a learned model adds useful predictive information.

In [ ]:
train_mean = y_train.mean()

baseline_valid_pred = np.full(len(y_valid), train_mean)
baseline_test_pred = np.full(len(y_test), train_mean)

baseline_valid_mae = mean_absolute_error(y_valid, baseline_valid_pred)
baseline_valid_rmse = np.sqrt(mean_squared_error(y_valid, baseline_valid_pred))
baseline_test_mae = mean_absolute_error(y_test, baseline_test_pred)
baseline_test_rmse = np.sqrt(mean_squared_error(y_test, baseline_test_pred))

print(f"Training mean rent: {train_mean:.2f}")
print(f"Validation baseline MAE:  {baseline_valid_mae:.2f}")
print(f"Validation baseline RMSE: {baseline_valid_rmse:.2f}")
print(f"Test baseline MAE:        {baseline_test_mae:.2f}")
print(f"Test baseline RMSE:       {baseline_test_rmse:.2f}")

## Task 3.1 — Interpret the Baseline

Answer:

1. What value does the baseline predict for every apartment?
2. Did the baseline use any feature values?
3. Why is this still a useful reference?
4. Why must the mean be calculated from `y_train` rather than all target values?
5. Which metric, MAE or RMSE, reacts more strongly to a few very large errors?

# Part IV — Simple Linear Regression

We first use only apartment area:

$$
\hat{y}=w_0+w_1(\text{area}).
$$

This lets us interpret the model visually and mathematically.

## Task 4.1 — Predict the Sign of the Slope

Before fitting the model:

1. Should $w_1$ be positive or negative?
2. What would a positive slope mean in this context?
3. Why should you not assume the relationship is perfectly linear?

**Your prediction:**

In [ ]:
simple_model = LinearRegression()

X_train_area = X_train[["area_m2"]]
X_valid_area = X_valid[["area_m2"]]
X_test_area = X_test[["area_m2"]]

simple_model.fit(X_train_area, y_train)

print("Intercept:", round(simple_model.intercept_, 3))
print("Slope for area_m2:", round(simple_model.coef_[0], 3))

## Task 4.2 — Interpret the Equation

Write the learned equation:

$$
\widehat{\text{rent}}
=
w_0+w_1(\text{area}).
$$

Then explain:

1. what the slope means in SAR per square meter;
2. what the intercept mathematically represents;
3. why the intercept may have weak real-world meaning if area $=0$ is outside the observed data range.

In [ ]:
simple_valid_pred = simple_model.predict(X_valid_area)

simple_valid_mae = mean_absolute_error(y_valid, simple_valid_pred)
simple_valid_mse = mean_squared_error(y_valid, simple_valid_pred)
simple_valid_rmse = np.sqrt(simple_valid_mse)
simple_valid_r2 = r2_score(y_valid, simple_valid_pred)

print(f"Simple Linear Regression — Validation MAE:  {simple_valid_mae:.2f}")
print(f"Simple Linear Regression — Validation MSE:  {simple_valid_mse:.2f}")
print(f"Simple Linear Regression — Validation RMSE: {simple_valid_rmse:.2f}")
print(f"Simple Linear Regression — Validation R^2:  {simple_valid_r2:.3f}")

## Task 4.3 — Compare with the Baseline

Complete the interpretation:

| Model | Validation MAE | Validation RMSE | Better than baseline? |
|---|---:|---:|---|
| Mean baseline |  |  |  |
| Area-only linear regression |  |  |  |

Answer:

1. Did using `area_m2` improve prediction?
2. Which metric improved more noticeably?
3. What does a positive $R^2$ mean relative to predicting the mean?
4. Would this one-feature model be sufficient for deployment? Why or why not?

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_train_area["area_m2"], y_train, alpha=0.65, label="Training data")

x_line = np.linspace(
    X_train_area["area_m2"].min(),
    X_train_area["area_m2"].max(),
    200
).reshape(-1, 1)

y_line = simple_model.predict(x_line)

plt.plot(x_line[:, 0], y_line, linewidth=2, label="Fitted line")
plt.xlabel("Area (m²)")
plt.ylabel("Monthly Rent")
plt.title("Simple Linear Regression: Rent vs. Area")
plt.legend()
plt.show()

## Task 4.4 — Visual Interpretation

From the plot:

1. Does rent generally increase with area?
2. Are apartments with similar area always priced similarly?
3. What omitted variables might explain vertical spread?
4. Is there evidence that one straight line cannot explain every observation?

# Part V — Multiple Linear Regression

Now use all five features:

$$
\hat{y}
=
w_0
+w_1x_1
+w_2x_2
+w_3x_3
+w_4x_4
+w_5x_5.
$$

This allows the model to adjust rent using more information.

In [ ]:
multi_model = LinearRegression()
multi_model.fit(X_train, y_train)

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": multi_model.coef_,
})

print("Intercept:", round(multi_model.intercept_, 3))
display(coef_table.round(3))

## Task 5.1 — Interpret Coefficient Signs

For each feature, predict the expected sign before using the learned values:

| Feature | Expected sign | Learned sign | Interpretation |
|---|---|---|---|
| `area_m2` |  |  |  |
| `bedrooms` |  |  |  |
| `building_age` |  |  |  |
| `distance_km` |  |  |  |
| `furnished` |  |  |  |

Then answer:

> Why should coefficient magnitudes not be compared casually when features use different units?

In [ ]:
multi_train_pred = multi_model.predict(X_train)
multi_valid_pred = multi_model.predict(X_valid)

def regression_report(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
    }

reports = pd.DataFrame({
    "Baseline_Validation": {
        "MAE": baseline_valid_mae,
        "MSE": mean_squared_error(y_valid, baseline_valid_pred),
        "RMSE": baseline_valid_rmse,
        "R2": r2_score(y_valid, baseline_valid_pred),
    },
    "Simple_Train": regression_report(y_train, simple_model.predict(X_train_area)),
    "Simple_Validation": regression_report(y_valid, simple_valid_pred),
    "Multiple_Train": regression_report(y_train, multi_train_pred),
    "Multiple_Validation": regression_report(y_valid, multi_valid_pred),
}).T

display(reports.round(3))

## Task 5.2 — Compare Models

Answer:

1. Which model has the lowest validation MAE?
2. Which model has the lowest validation RMSE?
3. How much does the multiple model improve over the mean baseline?
4. Is the training error lower than validation error?
5. Does the train/validation gap suggest severe overfitting?
6. Why is validation performance more important than training performance for model selection?

# Part VI — Manual Prediction

A trained linear model is still just an equation.

For one apartment:

$$
\hat{y}
=
b+\sum_{j=1}^{p}w_jx_j.
$$

You should be able to compute one prediction by hand from the learned coefficients.

In [ ]:
# Select one personalized validation example.
manual_index = X_valid.index[SEED % len(X_valid)]

x_manual = X_valid.loc[manual_index]
y_manual = y_valid.loc[manual_index]

print("Selected validation row index:", manual_index)
display(x_manual.to_frame("value"))
print("Actual monthly rent:", y_manual)

## Task 6.1 — Compute the Prediction by Hand

Using the model intercept and coefficient table, calculate:

$$
\hat{y}
=
b
+w_1(\text{area})
+w_2(\text{bedrooms})
+w_3(\text{age})
+w_4(\text{distance})
+w_5(\text{furnished}).
$$

Show each contribution separately.

Then compute the residual:

$$
e=y-\hat{y}.
$$

State whether the model **overpredicted** or **underpredicted**.

In [ ]:
library_prediction = multi_model.predict(
    pd.DataFrame([x_manual], columns=feature_cols)
)[0]

library_residual = y_manual - library_prediction

print(f"Library prediction: {library_prediction:.2f}")
print(f"Actual rent:        {y_manual:.2f}")
print(f"Residual y-yhat:    {library_residual:.2f}")

## Task 6.2 — Verify Your Manual Work

Compare your manual prediction with the library prediction.

1. Is the difference only due to rounding?
2. If not, identify your arithmetic error.
3. If the residual is positive, did the model overpredict or underpredict?
4. Why is manual verification useful even when software can calculate the answer automatically?

# Part VII — Residual Analysis

A residual is:

$$
e_i=y_i-\hat{y}_i.
$$

A good regression model should not leave an obvious systematic pattern in the residuals.

Residual plots can reveal:

- nonlinearity;
- changing variance;
- unusual observations;
- systematic overprediction or underprediction.

In [ ]:
valid_residuals = y_valid.to_numpy() - multi_valid_pred

residual_df = X_valid.copy()
residual_df["actual_rent"] = y_valid.to_numpy()
residual_df["predicted_rent"] = multi_valid_pred
residual_df["residual"] = valid_residuals
residual_df["abs_residual"] = np.abs(valid_residuals)

display(residual_df.head())

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(multi_valid_pred, valid_residuals, alpha=0.75)
plt.axhline(0, linewidth=1)
plt.xlabel("Predicted Rent")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residuals vs. Predicted Rent")
plt.show()

## Task 7.1 — Residual Plot Interpretation

Answer:

1. Are residuals centered approximately around zero?
2. Do you see a visible curve or systematic pattern?
3. Does the spread appear constant across predicted rents?
4. Are there unusually large residuals?
5. What could a curved residual pattern suggest?
6. What could a funnel-shaped pattern suggest?

In [ ]:
largest_errors = residual_df.sort_values(
    "abs_residual",
    ascending=False
).head(8)

display(largest_errors)

## Task 7.2 — Analyze Large Errors

Choose the two largest absolute residuals.

For each:

1. state the actual rent;
2. state the predicted rent;
3. state the residual;
4. identify which feature values look unusual;
5. propose one reason the linear model may have failed.

### Important

Do not assume a large residual means the row is invalid. It may represent a real high-end or unusual apartment.

## Task 7.3 — Residuals vs. Building Age

The synthetic data contain a mild nonlinear age effect.

Predict what you may see when residuals are plotted against `building_age`.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    residual_df["building_age"],
    residual_df["residual"],
    alpha=0.75
)
plt.axhline(0, linewidth=1)
plt.xlabel("Building Age")
plt.ylabel("Residual")
plt.title("Residuals vs. Building Age")
plt.show()

## Task 7.4 — Diagnose Model Limitation

Answer:

1. Do residuals for older buildings show a systematic tendency?
2. If yes, what does that suggest about the strictly linear age term?
3. What feature-engineering idea could help?
4. Why should that modification be evaluated on validation data rather than accepted automatically?

> You will study polynomial features and regularization more deeply in the next lab.

# Part VIII — Metric Reasoning

The four main regression metrics used here answer different questions.

$$
MAE=\frac{1}{n}\sum_i |y_i-\hat{y}_i|
$$

$$
MSE=\frac{1}{n}\sum_i (y_i-\hat{y}_i)^2
$$

$$
RMSE=\sqrt{MSE}
$$

$$
R^2=
1-
\frac{\sum_i(y_i-\hat{y}_i)^2}
{\sum_i(y_i-\bar{y})^2}
$$

## Task 8.1 — Complete a Metric Function

Complete the function so that it returns all four metrics.

In [ ]:
def my_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # TODO: compute the four metrics manually.
    mae = None
    mse = None
    rmse = None
    r2 = None

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
    }

In [ ]:
toy_y = np.array([10.0, 20.0, 30.0])
toy_pred = np.array([12.0, 18.0, 33.0])

toy_metrics = my_regression_metrics(toy_y, toy_pred)

assert abs(toy_metrics["MAE"] - (7/3)) < 1e-9
assert abs(toy_metrics["MSE"] - (17/3)) < 1e-9
assert abs(toy_metrics["RMSE"] - np.sqrt(17/3)) < 1e-9

print("Manual metric function tests passed.")
print(toy_metrics)

In [ ]:
manual_valid_metrics = my_regression_metrics(
    y_valid,
    multi_valid_pred
)

print("Your manual metric implementation:")
print(manual_valid_metrics)

print("\nScikit-learn reference:")
print(regression_report(y_valid, multi_valid_pred))

## Task 8.2 — Metric Interpretation

For the multiple regression model:

1. Interpret MAE in the target's original unit.
2. Interpret RMSE in the target's original unit.
3. Why is MSE harder to communicate directly?
4. Why is RMSE usually larger than or equal to MAE?
5. What does the validation $R^2$ tell you?
6. Which metric would you report to a property manager and why?

# Part IX — Deliberate Debugging

A student writes:

```python
model = LinearRegression()
model.fit(X_valid, y_valid)
pred = model.predict(X_valid)
```

and reports the resulting validation score.

This is an invalid evaluation procedure.

## Task 9.1 — Explain the Error

Answer:

1. Which dataset was used to learn the parameters?
2. Which dataset was used to evaluate the same parameters?
3. Why is the reported score optimistic?
4. What should the model be fitted on?
5. What should the validation set be used for?

## Task 9.2 — Correct the Workflow

Complete:

In [ ]:
debug_model = LinearRegression()

# TODO: fit on the correct split.
# debug_model.fit(...)

# TODO: predict on validation data.
debug_valid_pred = None

In [ ]:
if debug_valid_pred is None:
    raise ValueError("Complete the corrected workflow.")

assert len(debug_valid_pred) == len(y_valid)

correct_debug_mae = mean_absolute_error(y_valid, debug_valid_pred)
print(f"Correct validation MAE: {correct_debug_mae:.2f}")

# Part X — Personalized Model Challenge

Your student-ID seed assigns one feature to remove from the multiple model.

Before running the experiment, predict how validation performance will change.

In [ ]:
challenge_features = [
    "area_m2",
    "bedrooms",
    "building_age",
    "distance_km",
    "furnished",
]

removed_feature = challenge_features[SEED % len(challenge_features)]

print("Your assigned feature to remove:", removed_feature)

## Task 10.1 — Prediction Before Modification

Before fitting the modified model:

1. Do you expect MAE to increase, decrease, or stay similar?
2. Do you expect $R^2$ to increase, decrease, or stay similar?
3. Why?
4. Is your assigned feature likely redundant with another feature?

**Your prediction:**

In [ ]:
reduced_features = [
    f for f in feature_cols
    if f != removed_feature
]

reduced_model = LinearRegression()
reduced_model.fit(
    X_train[reduced_features],
    y_train
)

reduced_valid_pred = reduced_model.predict(
    X_valid[reduced_features]
)

reduced_report = regression_report(
    y_valid,
    reduced_valid_pred
)

print("Removed feature:", removed_feature)
print("Original multiple-model validation metrics:")
print(regression_report(y_valid, multi_valid_pred))
print("\nReduced-model validation metrics:")
print(reduced_report)

## Task 10.2 — Analyze the Modification

Compare your prediction with the result.

Answer:

1. Did validation MAE improve or worsen?
2. Did validation $R^2$ improve or worsen?
3. Was the feature more important than you expected?
4. Could correlated features hide the effect of removing one feature?
5. Would you make the final feature-selection decision from this one split only? Why or why not?

# Part XI — Final Test Evaluation

Only now, after the model structure has been studied using training and validation data, evaluate the selected **multiple linear regression model** on the final test set.

Do not tune the model after seeing this result.

In [ ]:
final_test_pred = multi_model.predict(X_test)

final_report = regression_report(y_test, final_test_pred)

print("Final test metrics:")
for key, value in final_report.items():
    print(f"{key}: {value:.3f}")

## Task 11.1 — Final Generalization Statement

Write a short final statement that includes:

- test MAE;
- test RMSE;
- test $R^2$;
- whether the model clearly beats the mean baseline;
- one important limitation revealed by residual analysis.

Do not say merely “the model is good.” Interpret the values in the context of apartment rent.

# Individual Understanding Check

Your instructor may select one question for a 60–90 second explanation.

1. What does the slope mean in simple linear regression?
2. Explain the difference between prediction error and residual.
3. Why does RMSE react more strongly than MAE to large errors?
4. What does $R^2=0$ mean?
5. Why can a residual plot reveal a problem that MAE cannot?
6. Why is a large residual not automatically a bad data row?
7. Why should the test set not guide feature selection?
8. For your personalized removed feature, explain what happened and why.

You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. What did the mean baseline contribute to this lab?
2. Why did multiple regression usually outperform area-only regression?
3. Which coefficient was easiest to interpret and why?
4. What was the most important pattern in the residual plots?
5. What is one limitation of a strictly linear model for this dataset?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that your notebook contains:

- [ ] your own student-ID-derived dataset;
- [ ] regression problem formulation;
- [ ] train/validation/test split reasoning;
- [ ] mean baseline analysis;
- [ ] simple linear regression fit and interpretation;
- [ ] multiple linear regression fit and coefficient interpretation;
- [ ] manual prediction and residual;
- [ ] residual-vs-prediction analysis;
- [ ] residual-vs-age analysis;
- [ ] completed manual metric function;
- [ ] corrected debugging workflow;
- [ ] personalized feature-removal experiment;
- [ ] final test evaluation;
- [ ] reflection answers;
- [ ] all required code cells executed successfully.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Algorithmic / modeling justification | **3** |
| Experimental analysis | **2** |
| Trace / prediction / debugging | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |

### Marking emphasis

Full marks require showing that you understand:

$$
\text{features}
\rightarrow
\text{linear equation}
\rightarrow
\text{prediction}
\rightarrow
\text{residual}
\rightarrow
\text{evaluation}
\rightarrow
\text{model limitation}.
$$

# Lab 4 Summary

You should now be able to analyze a complete linear-regression model:

$$
\boxed{
\hat{y}=w^\top x+b
}
$$

and connect it to:

- baseline comparison;
- coefficient interpretation;
- residuals;
- MAE, MSE, RMSE, and $R^2$;
- validation and test discipline;
- evidence of nonlinearity or unusual cases.

**Next lab:** Gradient Descent, Polynomial Models, and Regularization.